Fase 3 — Semana 2

## Núcleo algorítmico y programación orientada a objetos

**Caso de trabajo: Comodidades, capacidad y ubicación vs. precio y ocupación de los alojamientos Airbnb en Santiago de Chile**

---



---
## Configuración

Se configura el siguiente código para adaptarlo a nuestro dataset. Se recuerda leer README para la obtención del archivo crudo.

In [1]:
# =====================================================================
# CONFIGURACIÓN DEL PROYECTO
# Editen solo esta celda para trabajar con su propio conjunto de datos
# =====================================================================

# Ruta escrita desde la raíz del proyecto, no desde la carpeta de este cuaderno
RUTA_DATOS = "data/raw/datasetairbnb.csv"         # archivo crudo
RUTA_F2 = "data/processed/airbnb_procesado.csv"   # resultado de la F2, para comparar

COLUMNA_ID = "id"
OBJETIVOS = ["price", "estimated_occupancy_l365d"]   # no se imputan ni se escalan

VARIABLES_ESTUDIO = ["price", "estimated_occupancy_l365d", "amenities", "accommodates",
                     "bedrooms", "beds", "neighbourhood_cleansed", "latitude",
                     "host_is_superhost", "minimum_nights", "maximum_nights"]

COLUMNAS_IMPUTAR = ["accommodates", "latitude", "bedrooms", "beds",
                    "minimum_nights", "maximum_nights"]
COLUMNAS_MARCAR = ["bedrooms"]
COLUMNAS_NOMINALES = {"host_is_superhost": "superhost",
                      "neighbourhood_cleansed": "comuna"}
COLUMNAS_ESCALAR = ["accommodates", "latitude", "bedrooms", "beds",
                    "minimum_nights", "maximum_nights", "n_comodidades"]

COMODIDADES = {
    "tiene_aire_acondicionado": r"air conditioning|\bac\b",
    "tiene_estacionamiento":    r"parking",
    "tiene_piscina":            r"\bpool\b(?! table)",
    "tiene_lavadora":           r"\bwasher\b",
    "tiene_gimnasio":           r"\bgym\b",
    "tiene_ascensor":           r"elevator",
    "tiene_espacio_trabajo":    r"dedicated workspace",
    "tiene_self_checkin":       r"self check-in",
}

UMBRAL_PRECIO = 10_000_000
MAX_NOCHES = 1125
SEMILLA = 42
PROPORCION_PRUEBA = 0.2

COLUMNAS_ESPERADAS = [COLUMNA_ID, "room_type"] + VARIABLES_ESTUDIO

# La misma configuración, empaquetada para entregarla a los módulos de src/
CONFIG = {
    "objetivos": OBJETIVOS,
    "variables_estudio": VARIABLES_ESTUDIO,
    "columnas_imputar": COLUMNAS_IMPUTAR,
    "columnas_marcar": COLUMNAS_MARCAR,
    "columnas_nominales": COLUMNAS_NOMINALES,
    "columnas_escalar": COLUMNAS_ESCALAR,
    "comodidades": COMODIDADES,
    "umbral_precio": UMBRAL_PRECIO,
    "max_noches": MAX_NOCHES,
}

# ==== Rutas: esta parte es igual que antes ====
from pathlib import Path

def localizar_raiz(inicio=None):
    """Si el cuaderno está en F3, la raíz del proyecto es la carpeta de arriba."""
    inicio = Path.cwd() if inicio is None else Path(inicio)
    return inicio.parent if inicio.name in {"F1", "F2", "F3", "F4"} else inicio

RAIZ = localizar_raiz()
RUTA_DATOS = str(RAIZ / RUTA_DATOS)
RUTA_F2 = str(RAIZ / RUTA_F2)
DIR_PROCESADO = str(RAIZ / "data" / "processed")

print("Archivo crudo :", RUTA_DATOS)
print("Resultado F2  :", RUTA_F2)
print("Objetivos     :", OBJETIVOS)

Archivo crudo : c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\raw\datasetairbnb.csv
Resultado F2  : c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\processed\airbnb_procesado.csv
Objetivos     : ['price', 'estimated_occupancy_l365d']


## Preparación del entorno

Preparación del entorno requerido para la ejecución del cuaderno:

In [2]:
import os
import sys
import time
import timeit
import tracemalloc

import numpy as np
import pandas as pd

SEMILLA = 42
np.random.seed(SEMILLA)

# Los módulos del proyecto están en src/, en la raíz. Se agrega la raíz a la
# lista de carpetas donde Python busca módulos, para poder hacer "from src ..."
# OJO: si editan un archivo de src/, reinicien el kernel para que tome los cambios.
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

print("Python :", sys.executable)
print("pandas :", pd.__version__)
print("NumPy  :", np.__version__)
print("Semilla:", SEMILLA)

Python : c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\.venv\Scripts\python.exe
pandas : 3.1.1
NumPy  : 2.5.4
Semilla: 42


## Carga del conjunto desde el archivo externo

La función `cargar`, que vive en `src/carga.py`, hace tres cosas antes de devolver nada:

1. Comprueba que el archivo exista, y si no, explica dónde ponerlo.
2. Lo lee según su extensión, sea CSV o Excel.
3. Verifica que estén las columnas declaradas en la configuración.

In [3]:
# La carga vive en src/carga.py (leer_archivo, verificar_esquema y cargar).
# El cuaderno le entrega la ruta y las columnas: el módulo no guarda configuración.
from src.carga import cargar

datos = cargar(RUTA_DATOS, COLUMNAS_ESPERADAS)
datos.head()

Archivo leído: c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\raw\datasetairbnb.csv
Forma: 18904 filas x 90 columnas
Esquema verificado: las 13 columnas del pipeline están presentes


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,49392,https://www.airbnb.com/rooms/49392,20260923225049,2026-09-26,NaN,Share my Flat in Providencia,<b>The space</b><br />Departamento en el barri...,NaN,https://a0.muscache.com/pictures/3740612/b1850...,224592,...,NaN,NaN,NaN,NaN,NaN,1,0,1,0,NaN
1,52811,https://www.airbnb.com/rooms/52811,20260923225049,2026-09-26,NaN,Suite Providencia 1 Santiago Chile,Apartment located on the subway station Manuel...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,244792,...,4.59,4.64,4.36,NaN,NaN,3,3,0,0,0.23
2,80482,https://www.airbnb.com/rooms/80482,20260923225049,2026-09-25,NaN,Room Private for Woman,"Near transport, subway, malls supermarkets, wa...",NaN,https://a0.muscache.com/pictures/3633153/dffd5...,154527,...,NaN,NaN,NaN,NaN,NaN,2,0,2,0,NaN
3,88944,https://www.airbnb.com/rooms/88944,20260923225049,2026-09-26,NaN,COZY APT. PROVIDENCIA METRO WIFI TV,Spacious and cozy apartment. It has excellent ...,NaN,https://a0.muscache.com/pictures/8fd1c60d-5a8f...,485358,...,4.89,4.96,4.73,NaN,NaN,1,1,0,0,1.37
4,90694,https://www.airbnb.com/rooms/90694,20260923225049,2026-09-26,NaN,Apartment x Rent in Providencia 802,APARTMENT<br />Furnished Apartments with one b...,NaN,https://a0.muscache.com/pictures/d29b8d52-ca46...,491253,...,4.83,4.77,4.43,NaN,NaN,2,2,0,0,0.41


## Perfilar () el conjunto

Para perfilar el conjunto utilizado se define la función perfilar () que nos permitirá dar un primer vistazo al conjunto:

In [4]:
def perfilar(df):
    """Devuelve una fila por columna con su tipo, valores únicos y porcentaje de nulos."""
    return pd.DataFrame({
        "columna": df.columns,
        "tipo": [str(t) for t in df.dtypes],
        "unicos": [df[c].nunique(dropna=True) for c in df.columns],
        "nulos": df.isna().sum().values,
        "pct_nulos": (df.isna().mean() * 100).round(2).values,
    }).sort_values("pct_nulos", ascending=False).reset_index(drop=True)


perfil = perfilar(datos)

print("Columnas con valores faltantes:")
print(perfil[perfil["nulos"] > 0].to_string(index=False))

print("\nTop 10 columnas con más valores nulos:")
print(perfil[["columna", "pct_nulos"]].head(10).to_string(index=False))

print("\nDistribución del tipo de alojamiento:")
print(datos["room_type"].value_counts())

print("\nDistribución del tipo de propiedad:")
print(datos["property_type"].value_counts().head(10))

print("\nObjetivos del estudio:", OBJETIVOS)
print("Ojo: en el crudo 'price' viene como texto, por ejemplo:", datos["price"].dropna().iloc[0])
print(datos["estimated_occupancy_l365d"].describe())

Columnas con valores faltantes:
                     columna    tipo  unicos  nulos  pct_nulos
       neighborhood_overview float64       0  18904     100.00
                  host_since float64       0  18904     100.00
          host_response_time float64       0  18904     100.00
          host_thumbnail_url float64       0  18904     100.00
        host_acceptance_rate float64       0  18904     100.00
          host_response_rate float64       0  18904     100.00
          host_verifications float64       0  18904     100.00
               neighbourhood float64       0  18904     100.00
   host_total_listings_count float64       0  18904     100.00
          host_neighbourhood float64       0  18904     100.00
            calendar_updated float64       0  18904     100.00
            instant_bookable float64       0  18904     100.00
neighbourhood_group_cleansed float64       0  18904     100.00
                   bathrooms float64       0  18904     100.00
                     li

**Tres observaciones que condicionan todo lo que viene (documentadas en la Fase 2).**

1. `price` viene como texto (`$55,106.00`): hay que convertirlo a número antes de cualquier cálculo.
2. Los objetivos no se imputan. Rellenar `price` o `estimated_occupancy_l365d` sería inventar la respuesta: los alojamientos sin precio se eliminan.
3. Hay valores imposibles: precios sobre $10.000.000, `maximum_nights` de 2.147.483.647 y anuncios con más dormitorios que huéspedes.

Ninguna decisión se rehace aquí: la Fase 3 las encapsula en clases.

---
# 1. De funciones a objetos

Así se ve el código de la Fase 2, en celdas sueltas:

In [5]:
# El estilo de la Fase 2: funciona, pero cada celda depende de las anteriores
df_f2 = datos.copy()
df_f2 = df_f2.drop(columns=["id"])
df_f2["beds"] = df_f2["beds"].fillna(df_f2["beds"].median())
df_f2 = pd.get_dummies(df_f2, columns=["room_type"], dtype=int)

print("Resultado (Filas x Columnas):", df_f2.shape)

Resultado (Filas x Columnas): (18904, 92)


En este cuaderno se organiza el código basado en programación orientada a objetos a través de modularidad en src/ como se muestra a continuación:

![alt text](../docs/Organización_modular.png)

Cada uno de los módulos .py, contiene clases con métodos distintos que van desde cargar datos, aprender, transformar hasta realizar mediciones de velocidad y memoria. El detalle se describe en el informe asociado. 

---
# 2. Encapsulamiento

Se aplica el principio de encapsulamiento mediante las clases de transformación implementadas en src/transformadores.py.

Este principio permite organizar los parámetros internos de cada transformador y controlar cómo se utilizan durante el preprocesamiento de los datos.

Para ello, se distinguen dos operaciones fundamentales:

- ajustar(): aprende los parámetros necesarios a partir del conjunto de entrenamiento, como la mediana utilizada para imputar valores faltantes.

- transformar(): aplica los parámetros previamente aprendidos sobre los datos, sin recalcularlos a partir del conjunto de prueba.

Esta separación contribuye a prevenir la fuga de datos (data leakage), evitando que la información del conjunto de prueba influya en los parámetros aprendidos durante el entrenamiento.

Como parte de este procedimiento, el dataset de Airbnb se divide en conjuntos de entrenamiento y prueba mediante una proporción definida y una semilla aleatoria fija. 
Se ha dejado un ejemplo sobre el encapsulamiento:



In [11]:
class ImputadorCamas:
    """Imputa beds separando lo que aprende de lo que aplica."""

    def __init__(self):
        self._mediana = None        # interno: se aprende, no se asigna desde fuera
        self._ajustado = False      # interno: controla el orden de las llamadas

    @property
    def ajustado(self):
        """Solo lectura: se puede consultar, no asignar."""
        return self._ajustado

    @property
    def mediana(self):
        return self._mediana

    def ajustar(self, df):
        self._mediana = float(df["beds"].median())
        self._ajustado = True
        return self

    def transformar(self, df):
        if not self._ajustado:
            raise RuntimeError(
                "ImputadorCamas: hay que llamar a ajustar() antes de transformar(). "
                "La mediana se aprende del conjunto de entrenamiento."
            )
        df = df.copy()
        df["beds"] = df["beds"].fillna(self._mediana)
        return df


# El orden equivocado ahora falla con un mensaje claro
imputador = ImputadorCamas()
try:
    imputador.transformar(datos)
except RuntimeError as error:
    print("RuntimeError:", error)

RuntimeError: ImputadorCamas: hay que llamar a ajustar() antes de transformar(). La mediana se aprende del conjunto de entrenamiento.


In [12]:
# El orden correcto, con separación de conjuntos
entrenamiento = datos.sample(frac=1 - PROPORCION_PRUEBA, random_state=SEMILLA)
prueba = datos.drop(index=entrenamiento.index)

imputador = ImputadorCamas().ajustar(entrenamiento)
prueba_lista = imputador.transformar(prueba)

print("Mediana aprendida del entrenamiento:", round(imputador.mediana, 2))
print("Mediana del conjunto de prueba     :", round(prueba["beds"].median(), 2))
print("Nulos en prueba después de imputar :", int(prueba_lista["beds"].isna().sum()))
print("\nLa mediana aplicada a prueba es la del entrenamiento, aunque en este caso")
print("ambas coincidan: prueba nunca se usó para aprender nada.")

# La propiedad es de solo lectura
try:
    imputador.ajustado = False
except AttributeError as error:
    print("\nAttributeError:", error)

Mediana aprendida del entrenamiento: 2.0
Mediana del conjunto de prueba     : 2.0
Nulos en prueba después de imputar : 0

La mediana aplicada a prueba es la del entrenamiento, aunque en este caso
ambas coincidan: prueba nunca se usó para aprender nada.

AttributeError: property 'ajustado' of 'ImputadorCamas' object has no setter


**Por qué importa.** Sin el control de `_ajustado`, transformar antes de ajustar
produciría un resultado sin sentido **y sin ningún mensaje de error**. El
encapsulamiento convierte un error silencioso en uno visible.

En el informe, este bloque es la evidencia del encapsulamiento: atributos internos con
guion bajo, propiedades de solo lectura y validación antes de actuar.

---
# 3. Herencia y polimorfismo

En esta sección, se deja evidencia de cómo los métodos de la clase madre se heredan en las clases hijas.
La clase madre en cuestión es transformador y sus métodos se heredan en las clases hijas. Sin embargo, estos métodos son definidos según el propósito de la clase hija, demostrando de esa forma el concepto de polimorfismo.

Se importa la clase madre:

In [ ]:
# La clase madre vive en src/transformadores.py: el código completo, comentado
# línea por línea, está en ese archivo. Aquí se importa y se pone a prueba.
from src.transformadores import Transformador

# La clase base no se usa sola
try:
    Transformador("beds").ajustar(datos)
except NotImplementedError as error:
    print("NotImplementedError:", error)

NotImplementedError: Cada clase hija debe implementar aprender().


Se importan las clases hijas:

In [14]:
# Las clases hijas también viven en src/transformadores.py
from src.transformadores import (
    CodificadorNominal,
    EliminadorColumnas,
    EscaladorRobusto,
    ImputadorMediana,
)

imp = ImputadorMediana("beds")
salida = imp.ajustar_transformar(datos)
print(imp.nombre, "->", imp.parametros)
print("Nulos en beds después:", int(salida["beds"].isna().sum()))

ImputadorMediana(beds) -> {'mediana': 2.0, 'nulos_en_ajuste': 203}
Nulos en beds después: 0


**Se realiza la transformación de la variable** "price", eliminando interferencias que hacen ver al valor como str, transformando de esta manera a float.

In [15]:
# Los 6 pasos de limpieza de la Fase 2, desde src/transformadores.py
from src.transformadores import (
    Acotador,
    ConversorPrecio,
    ExtractorComodidades,
    FiltroFilas,
    MarcadorNulos,
    SelectorColumnas,
)

# Prueba rápida: el precio deja de ser texto
conversor = ConversorPrecio("price")
convertido = conversor.ajustar_transformar(datos)
print(conversor.nombre, "->", datos["price"].dropna().iloc[0], "pasa a",
      convertido["price"].dropna().iloc[0])

ConversorPrecio(price) -> $59,303.00 pasa a 59303.0


### Cómo comprobar que la herencia funciona

No hay que creer en la herencia: se puede verificar.

In [16]:
imputador = ImputadorMediana("beds")

# 1) El objeto ES un ImputadorMediana Y TAMBIÉN es un Transformador
print("¿Es ImputadorMediana?", isinstance(imputador, ImputadorMediana))
print("¿Es Transformador?   ", isinstance(imputador, Transformador))

# 2) La cadena de búsqueda muestra de dónde hereda
print("\\nCadena de herencia:", [c.__name__ for c in ImputadorMediana.__mro__])

# 3) Métodos que NO están escritos en la clase hija y sin embargo funcionan,
#    porque se heredaron de la clase base
print("\\nMétodos heredados de Transformador:")
for metodo in ["ajustar", "transformar", "ajustar_transformar"]:
    definido_en = "ImputadorMediana" if metodo in ImputadorMediana.__dict__ else "Transformador"
    print(f"  {metodo:<22} definido en {definido_en}")

print("\\nMétodos propios de la clase hija:")
for metodo in ["aprender", "aplicar"]:
    print(f"  {metodo:<22} definido en ImputadorMediana")

¿Es ImputadorMediana? True
¿Es Transformador?    True
\nCadena de herencia: ['ImputadorMediana', 'Transformador', 'object']
\nMétodos heredados de Transformador:
  ajustar                definido en Transformador
  transformar            definido en Transformador
  ajustar_transformar    definido en Transformador
\nMétodos propios de la clase hija:
  aprender               definido en ImputadorMediana
  aplicar                definido en ImputadorMediana


**Polimorfismo**: a continuación se muestran cinco transformaciones el dataset utilizando las clases hijas, se puede ver cómo al final el código realiza un bucle de ajustar_transformar, método que se describe en la clase madre:

In [17]:
pasos = [
    EliminadorColumnas("id"),
    ImputadorMediana("beds"),
    CodificadorNominal("room_type"),
    CodificadorNominal("neighbourhood_cleansed"),
    EscaladorRobusto("minimum_nights")
]

resultado = datos.copy()
for paso in pasos:                      # la misma llamada polimórfica para todos los pasos
    resultado = paso.ajustar_transformar(resultado)
    print(f"{paso.nombre:<45} -> {resultado.shape}")

EliminadorColumnas(id)                        -> (18904, 89)
ImputadorMediana(beds)                        -> (18904, 89)
CodificadorNominal(room_type)                 -> (18904, 92)
CodificadorNominal(neighbourhood_cleansed)    -> (18904, 123)
EscaladorRobusto(minimum_nights)              -> (18904, 123)


**Este bloque es la evidencia de los tres principios**

- **Herencia**: las cuatro clases heredan de `Transformador` y no repiten el control de estado.
- **Polimorfismo**: el bucle llama `ajustar_transformar()` sin preguntar el tipo.
- **Encapsulamiento**: `_parametros` y `_ajustado` son internos, y hay validación antes de actuar.

---
# 6. El `Pipeline`

Una clase que guarda el orden, los ejecuta y controla que se haya ajustado antes de
transformar. 
**Nuestro pipeline coordina la secuencia de transformaciones, contiene una lista de objetos Transformador, ejecuta los pasos en orden y resume/organiza el flujo del preprocesamiento.**

In [18]:
# El orquestador vive en src/pipeline.py
from src.pipeline import Pipeline

pipeline = Pipeline([
    EliminadorColumnas("id"),
    ImputadorMediana("beds"),
    CodificadorNominal("room_type"),
    CodificadorNominal("neighbourhood_cleansed"),
    EscaladorRobusto("minimum_nights"),
])

print(pipeline)
pipeline.resumen()

Pipeline(5 pasos, sin ajustar)


,orden,paso,clase
0,1,EliminadorColumnas(id),EliminadorColumnas
1,2,ImputadorMediana(beds),ImputadorMediana
2,3,CodificadorNominal(room_type),CodificadorNominal
3,4,CodificadorNominal(neighbourhood_cleansed),CodificadorNominal
4,5,EscaladorRobusto(minimum_nights),EscaladorRobusto


El código a continuación ajusta con entrenamiento y transforma prueba para evitar la fuga de datos, tal como se mencionó en el apartado de Encapsulamiento:

In [19]:
# Ajustar con entrenamiento, transformar prueba: sin fuga de datos
pipeline.ajustar(entrenamiento)
prueba_lista = pipeline.transformar(prueba)

print("Entrenamiento:", entrenamiento.shape, " -> Prueba transformada:", prueba_lista.shape)

escalador = pipeline.pasos_ejecutados()[-1]
print(f"\n{escalador.nombre} aprendió del entrenamiento: {escalador.parametros}")
print("Mediana de minimum_nights en prueba, ANTES de escalar:", prueba["minimum_nights"].median())
print("Esos parámetros se aplicaron a prueba sin recalcular nada con sus datos.")

Entrenamiento: (15123, 90)  -> Prueba transformada: (3781, 123)

EscaladorRobusto(minimum_nights) aprendió del entrenamiento: {'mediana': 1.0, 'ric': 1.0}
Mediana de minimum_nights en prueba, ANTES de escalar: 2.0
Esos parámetros se aplicaron a prueba sin recalcular nada con sus datos.


**Por qué importa.** El escalador no recalcula la mediana ni el RIC con los datos de prueba: aplica los que aprendió del entrenamiento. Si salieran de prueba, el análisis usaría información que no debía ver.

A continuación, se muestra cómo a través del módulo pipeline.py podemos orquestar las 24 transformaciones del dataset:

In [20]:
# construir_pipeline(config) vive en src/pipeline.py y arma los 24 pasos de la Fase 2.
from src.pipeline import construir_pipeline as construir_pipeline_desde_config


def construir_pipeline():
    """Atajo del cuaderno: el pipeline de src/ con la CONFIG de este notebook.

    Así el resto del cuaderno puede seguir escribiendo construir_pipeline()
    sin repetir la configuración en cada celda.
    """
    return construir_pipeline_desde_config(CONFIG)


pipeline_proyecto = construir_pipeline()
print(pipeline_proyecto)
pipeline_proyecto.resumen()

Pipeline(24 pasos, sin ajustar)


,orden,paso,clase
0,1,ConversorPrecio(price),ConversorPrecio
1,2,FiltroFilas(solo alojamientos completos),FiltroFilas
2,3,SelectorColumnas(11 variables),SelectorColumnas
3,4,Acotador(maximum_nights),Acotador
4,5,FiltroFilas(objetivos sin faltantes),FiltroFilas
5,6,MarcadorNulos(bedrooms),MarcadorNulos
6,7,ImputadorMediana(accommodates),ImputadorMediana
7,8,ImputadorMediana(latitude),ImputadorMediana
8,9,ImputadorMediana(bedrooms),ImputadorMediana
9,10,ImputadorMediana(beds),ImputadorMediana


---
# 7. Cohesión y acoplamiento

Son las dos palabras que más aparecen en el criterio de diseño estructurado y que casi
ningún material explica. Acá van con código.

**Alta cohesión** significa que cada componente hace **una sola cosa**.

**Bajo acoplamiento** significa que los componentes dependen poco unos de otros: cambiar
uno no obliga a tocar los demás.

## 7.1 Cohesión baja: una clase que hace de todo

Este es el diseño que aparece cuando se escribe rápido. Funciona, pero cada método toca
cosas distintas y la clase no tiene un propósito único.

In [21]:
class ProcesadorAirbnbTodoEnUno:
    """Ejemplo de COHESIÓN BAJA en el proyecto: hace cuatro cosas sin relación entre sí."""

    def __init__(self, ruta_csv):
        self.ruta_csv = ruta_csv
        self.df = None

    def cargar(self):            # Responsabilidad 1: entrada y salida de archivos (I/O)
        self.df = pd.read_csv(self.ruta_csv)

    def limpiar(self):           # Responsabilidad 2: transformar datos (Pipeline)
        if self.df is not None:
            self.df = self.df.dropna(subset=["beds"])

    def graficar(self):          # Responsabilidad 3: visualización (Matplotlib / Seaborn)
        # Generar gráficos de precios o distribución de alojamientos en Santiago
        pass

    def enviar_correo(self):     # Responsabilidad 4: comunicación / notificaciones
        # Enviar reporte de la auditoría de alojamientos por email
        pass


print("Cuatro responsabilidades distintas (cargar, limpiar, graficar, enviar correo) en una sola clase.")
print("Problema práctico: para probar la limpieza de los datos de Airbnb hay que tener un archivo real,")
print("porque cargar() y limpiar() viven pegadas en el mismo objeto.")

Cuatro responsabilidades distintas (cargar, limpiar, graficar, enviar correo) en una sola clase.
Problema práctico: para probar la limpieza de los datos de Airbnb hay que tener un archivo real,
porque cargar() y limpiar() viven pegadas en el mismo objeto.


## 7.2 Cohesión alta: cada clase con un propósito

El mismo trabajo, repartido. Ahora cada pieza se puede probar sola.

In [22]:
import pandas as pd
import numpy as np

class CargadorAirbnb:
    """Una sola responsabilidad: leer el archivo CSV con los alojamientos de Santiago."""

    def __init__(self, ruta):
        self.ruta = ruta

    def cargar(self):
        return pd.read_csv(self.ruta)


class LimpiadorAirbnb:
    """Una sola responsabilidad: limpiar nulos en métricas clave (como beds).

    Fíjense en que NO sabe de dónde vienen los datos. Recibe un DataFrame y
    devuelve otro. Eso es bajo acoplamiento: no depende del Cargador.
    """

    def limpiar(self, df):
        # Limpiar filas donde falte el puntaje de reseñas de los alojamientos
        df = df.copy()
        if "beds" in df.columns:
            df = df.dropna(subset=["beds"])
        return df


# La gran ventaja se ve al probar: no hace falta ningún archivo real ni CSV en disco
mini = pd.DataFrame({
    "id": [101, 102, 103],
    "beds": [4.5, np.nan, 4.8],
    "room_type": ["Entire home/apt", "Private room", "Entire home/apt"]
})

print("Se puede probar el LimpiadorAirbnb sin tocar el disco ni archivos pesados:")
limpiador = LimpiadorAirbnb()
print("  entrada:", mini.shape, " -> salida:", limpiador.limpiar(mini).shape)

Se puede probar el LimpiadorAirbnb sin tocar el disco ni archivos pesados:
  entrada: (3, 3)  -> salida: (2, 3)


## 7.3 La prueba del acoplamiento

Hay una pregunta que resuelve el asunto sin teoría:

> **Si cambiamos la forma de imputar, ¿cuántos archivos hay que modificar?**

Si la respuesta es **uno**, el acoplamiento es bajo. Si son tres, el diseño es frágil.

Veámoslo con el pipeline que ya construimos.

In [23]:
# Cambiar un paso del pipeline no obliga a tocar ni el Pipeline ni los otros pasos
pipeline_a = Pipeline([EliminadorColumnas("id"), ImputadorMediana("beds")])
pipeline_b = Pipeline([EliminadorColumnas("id"), ImputadorMediana("minimum_nights")])

for nombre, pipe in [("versión A", pipeline_a), ("versión B", pipeline_b)]:
    salida = pipe.ajustar(datos).transformar(datos)
    print(f"{nombre}: {salida.shape}")

print("\nSe cambió el paso y NO se modificó la clase Pipeline ni las demás clases.")
print("Eso es bajo acoplamiento, y es lo que el criterio de diseño evalúa.")

versión A: (18904, 89)
versión B: (18904, 89)

Se cambió el paso y NO se modificó la clase Pipeline ni las demás clases.
Eso es bajo acoplamiento, y es lo que el criterio de diseño evalúa.


## 7.4 Cómo se traduce en la estructura de archivos

Una organización con alta cohesión y bajo acoplamiento se reconoce a simple vista:

```
src/
├── carga.py            solo lee y verifica archivos
├── transformadores.py  solo las clases de transformación
├── pipeline.py         solo encadena y ejecuta
└── medicion.py         solo compara implementaciones
```

En el proyecto modelo, estos cuatro archivos existen en la carpeta `src/` de la raíz y
contienen las mismas clases y funciones que este cuaderno desarrolla paso a paso. La
sección 13 verifica que ambas versiones producen el mismo resultado.

Cada archivo tiene un propósito que se puede enunciar en una frase. Si al describir un
archivo hace falta la palabra «y» varias veces, probablemente tenga cohesión baja.

**Para el informe.** El apartado de diseño estructurado pide justificar la organización.
La prueba del acoplamiento es el mejor argumento: «para cambiar la estrategia de
imputación solo se modifica `transformadores.py`, porque el pipeline no conoce el
detalle de cada paso».

---
# 8. Validación: casos normales, límite y excepciones

La rúbrica pide los tres escenarios. Acá va uno de cada tipo, con el conjunto de Airbnb.

In [24]:
# --- CASO NORMAL: el flujo completo sobre el conjunto real ---
pipe = Pipeline([EliminadorColumnas("id"), ImputadorMediana("beds")])
salida = pipe.ajustar(datos).transformar(datos)

assert len(salida) == len(datos), "El pipeline perdió o duplicó filas"
assert salida["beds"].isna().sum() == 0, "Quedaron nulos en beds"
assert "id" not in salida.columns, "La columna id no se eliminó"
print("Caso normal: las tres comprobaciones pasaron.")

Caso normal: las tres comprobaciones pasaron.


In [25]:
# --- CASOS LÍMITE: situaciones extremas pero válidas ---

# 1) Una columna sin ningún nulo no debe alterarse
sin_nulos = pd.DataFrame({"beds": [4.5, 4.8, 5.0]})
r1 = ImputadorMediana("beds").ajustar_transformar(sin_nulos)
assert r1["beds"].tolist() == [4.5, 4.8, 5.0]
print("Límite 1: columna sin nulos, sin cambios.")

# 2) Una sola categoría genera una sola columna
una_cat = pd.DataFrame({"room_type": ["Entire home/apt", "Entire home/apt", "Entire home/apt"]})
r2 = CodificadorNominal("room_type").ajustar_transformar(una_cat)
print("Límite 2: una categoría ->", list(r2.columns))

# 3) Varianza cero no debe producir división por cero
constante = pd.DataFrame({"minimum_nights": [10.0, 10.0, 10.0]})
r3 = EscaladorRobusto("minimum_nights").ajustar_transformar(constante)
assert r3["minimum_nights"].notna().all()
print("Límite 3: varianza cero ->", r3["minimum_nights"].tolist())

# 4) Una categoría nueva en prueba no debe crear columna
pipe_cat = Pipeline([CodificadorNominal("neighbourhood_cleansed")]).ajustar(entrenamiento)
nuevos = prueba.copy()
# Insertamos una comuna que no existía en el conjunto de entrenamiento
nuevos.iloc[0, nuevos.columns.get_loc("neighbourhood_cleansed")] = "Comuna Ficticia"
r4 = pipe_cat.transformar(nuevos)
print("Límite 4: categoría nueva no crea columna ->",
      "neighbourhood_cleansed_Comuna_Ficticia" not in r4.columns)

Límite 1: columna sin nulos, sin cambios.
Límite 2: una categoría -> ['room_type_Entire_home/apt']
Límite 3: varianza cero -> [0.0, 0.0, 0.0]
Límite 4: categoría nueva no crea columna -> True


In [26]:
# --- EXCEPCIONES: entradas que deben fallar con mensaje claro ---

pruebas = [
    ("transformar sin ajustar", lambda: ImputadorMediana("beds").transformar(datos)),
    ("columna inexistente", lambda: ImputadorMediana("columna_ficticia").ajustar(datos)),
    ("agregar algo que no es Transformador", lambda: Pipeline().agregar("texto")),
    ("pipeline sin ajustar", lambda: Pipeline([ImputadorMediana("beds")]).transformar(datos)),
]

for descripcion, accion in pruebas:
    try:
        accion()
        print(f"  {descripcion:<38} NO lanzó excepción (revisar)")
    except (RuntimeError, KeyError, TypeError) as error:
        print(f"  {descripcion:<38} {type(error).__name__} capturado")

  transformar sin ajustar                RuntimeError capturado
  columna inexistente                    KeyError capturado
  agregar algo que no es Transformador   TypeError capturado
  pipeline sin ajustar                   RuntimeError capturado


## La verificación que más rinde: el pipeline de clases contra la Fase 2

Si el pipeline con clases produce exactamente el mismo conjunto que guardó la Fase 2, hay una prueba objetiva de que la reorganización no rompió nada. Se compara contra `data/processed/airbnb_procesado.csv`, ajustando con todos los datos porque así trabajó la Fase 2. El índice se reinicia porque el One-Hot de la Fase 2 lo reinició.

In [27]:
# Resultado oficial de la Fase 2, tal como quedó guardado en el repositorio
f2_oficial = pd.read_csv(RUTA_F2)

# El mismo trabajo, hecho por el pipeline de clases.
# Se ajusta con TODOS los datos porque así trabajó la F2 (sin separar entrenamiento y prueba)
pipeline_f2 = construir_pipeline().ajustar(datos)
f3_clases = pipeline_f2.transformar(datos).reset_index(drop=True)

print("Fase 2 (archivo guardado):", f2_oficial.shape)
print("Fase 3 (pipeline de clases):", f3_clases.shape)

# Compara filas, columnas, nombres, orden, tipos y valores.
# Si hubiera UNA sola diferencia, se detiene con un error que dice dónde está.
pd.testing.assert_frame_equal(f3_clases, f2_oficial)
print("\nLas dos versiones son idénticas: mismas filas, columnas, tipos y valores.")

Fase 2 (archivo guardado): (14758, 50)
Fase 3 (pipeline de clases): (14758, 50)

Las dos versiones son idénticas: mismas filas, columnas, tipos y valores.


---
# 9. Recursividad

Toda función recursiva tiene dos partes: un **caso base** que la detiene y un **caso
recursivo** que reduce el problema y vuelve a llamarse.

**Cuándo se justifica:** cuando no se sabe de antemano cuán profundo es el problema.
Si la profundidad es fija, un bucle es más simple y más rápido.

In [28]:
def aplanar(estructura, prefijo=""):
    """Convierte metadatos anidados en pares plano de clave y valor.

    Por qué recursión y no un bucle: la profundidad no se conoce al escribir el
    código. Mañana alguien agrega un nivel y el bucle anidado deja de servir.
    """
    plano = {}
    for clave in estructura:
        valor = estructura[clave]
        compuesta = f"{prefijo}.{clave}" if prefijo else str(clave)
        if isinstance(valor, dict) and valor:
            plano.update(aplanar(valor, compuesta))
        else:
            plano[compuesta] = valor
    return plano


# Metadatos adaptados al proyecto de Airbnb
metadatos = {
    "proyecto": {"nombre": "airbnb_santiago", "fase": 3},
    "datos": {"filas": len(datos), "nulos": {"beds": int(datos["beds"].isna().sum())}},
    "entorno": {"semilla": SEMILLA, "librerias": {"pandas": pd.__version__}},
}

for clave, valor in aplanar(metadatos).items():
    print(f"{clave:<32} {valor}")

proyecto.nombre                  airbnb_santiago
proyecto.fase                    3
datos.filas                      18904
datos.nulos.beds                 203
entorno.semilla                  42
entorno.librerias.pandas         3.1.1


## El costo de repetir trabajo

La recursión **no es** sinónimo de eficiencia. Estas dos funciones tienen la misma
idea y un costo muy distinto.

In [29]:
import time
import pandas as pd

def fib_ingenua(n):
    """Sin memoria: recalcula los mismos subproblemas miles de veces."""
    if n < 2:
        return n
    return fib_ingenua(n - 1) + fib_ingenua(n - 2)


def fib_memoizada(n, cache=None):
    """La misma recursión, guardando lo ya calculado."""
    if cache is None:
        cache = {}
    if n in cache:
        return cache[n]
    resultado = n if n < 2 else fib_memoizada(n - 1, cache) + fib_memoizada(n - 2, cache)
    cache[n] = resultado
    return resultado


filas = []
for n in [18, 22, 26]:
    inicio = time.perf_counter(); fib_ingenua(n); t1 = time.perf_counter() - inicio
    inicio = time.perf_counter(); fib_memoizada(n); t2 = time.perf_counter() - inicio
    filas.append({
        "n": n, 
        "ingenua_s": round(t1, 6), 
        "memoizada_s": round(t2, 6),
        "veces_mas_rapida": round(t1 / t2, 1) if t2 > 0 else 0.0
    })

pd.DataFrame(filas)

,n,ingenua_s,memoizada_s,veces_mas_rapida
0,18,0.000354,0.000006,57.1
1,22,0.002839,0.000009,305.3
2,26,0.017562,0.000014,1272.6


El cambio entre una versión y otra es de dos líneas, y el efecto se mide en cientos de
veces. Lo que hace eficiente a la segunda no es la recursión: es **no repetir trabajo**.

**Si su proyecto no tiene un problema recursivo natural**, decláren­lo en el informe y
expliquen por qué un enfoque iterativo es preferible. La rúbrica acepta recursividad
**o** división funcional, y una decisión argumentada vale lo mismo que implementarla.

---
# 10. Eficiencia: medir tiempo y memoria

La rúbrica pide mediciones **reproducibles** con `timeit` o equivalente, comparación
entre implementaciones e interpretación considerando tiempo **y** memoria.

In [30]:
# medir (tiempo y memoria pico) vive en src/medicion.py
from src.medicion import medir


# Datos reales justo antes de escalar: todos los pasos del pipeline menos los escaladores
pasos_previos = [p for p in construir_pipeline().pasos_ejecutados()
                 if not isinstance(p, EscaladorRobusto)]
antes_de_escalar = datos.copy()
for paso in pasos_previos:
    antes_de_escalar = paso.ajustar_transformar(antes_de_escalar)

escalador = EscaladorRobusto("n_comodidades").ajustar(antes_de_escalar)
MEDIANA, RIC = escalador.parametros["mediana"], escalador.parametros["ric"]


def escalar_con_bucle(serie):
    """Versión A: valor por valor; devuelve una Serie, igual que EscaladorRobusto."""
    escalados = []
    for valor in serie:
        escalados.append((valor - MEDIANA) / RIC)
    return pd.Series(escalados, index=serie.index, name=serie.name)


def escalar_vectorizado(serie):
    """Versión B: la columna completa de una vez (la que usa EscaladorRobusto)."""
    return (serie - MEDIANA) / RIC





serie = antes_de_escalar["n_comodidades"]

# Antes de comparar tiempos: las dos deben dar EXACTAMENTE lo mismo
pd.testing.assert_series_equal(escalar_con_bucle(serie), escalar_vectorizado(serie))
print(f"Las dos versiones dan el mismo resultado sobre los {len(serie)} alojamientos.\n")

# Se mide 5 veces y se conserva el MENOR tiempo
t_bucle = min(timeit.repeat(lambda: escalar_con_bucle(serie), number=50, repeat=5)) / 50
t_vect = min(timeit.repeat(lambda: escalar_vectorizado(serie), number=50, repeat=5)) / 50
_, _, mem_bucle = medir(escalar_con_bucle, serie)
_, _, mem_vect = medir(escalar_vectorizado, serie)

print(f"A · con bucle   : {t_bucle*1000:.3f} ms   {mem_bucle*1024:.1f} KB")
print(f"B · vectorizada : {t_vect*1000:.3f} ms   {mem_vect*1024:.1f} KB")
print(f"\nLa vectorizada es {t_bucle / t_vect:.1f} veces más rápida "
      f"y usa {mem_vect / mem_bucle:.2f} veces la memoria del bucle.")

Las dos versiones dan el mismo resultado sobre los 14758 alojamientos.

A · con bucle   : 2.162 ms   1299.7 KB
B · vectorizada : 0.075 ms   233.4 KB

La vectorizada es 28.8 veces más rápida y usa 0.18 veces la memoria del bucle.


**Cómo se lee.** Las dos versiones hacen la misma cuenta, `(valor − mediana) / RIC`, y `assert_series_equal` confirma que el resultado es idéntico. La diferencia está en **quién** hace el trabajo:

- **Con bucle**, Python ejecuta una resta y una división por cada alojamiento, una por una, y guarda cada resultado como un objeto `float` de Python en una lista que después convierte en Serie.
- **Vectorizada**, pandas entrega la columna completa a NumPy, que hace todas las cuentas de una vez en código compilado y guarda cada valor en 8 bytes.

**La vectorizada gana en las dos dimensiones.** Es unas 28 veces más rápida y su pico de memoria (233 KB) equivale a dos arreglos de 14.758 valores: el intermedio de la resta y el resultado. El bucle necesita cerca de cinco veces más memoria, porque cada valor es un objeto de Python y además se copia a la Serie final.

**Corrección respecto del avance formativo.** En la formativa concluimos que la versión vectorizada usaba un 24 % más de memoria. Esa medición incluía la conversión final a lista (`.tolist()`), que `EscaladorRobusto` no hace. Al devolver una Serie en ambas versiones, igual que el código de `src/`, la conclusión se invierte.

**Cómo crece el costo con el tamaño.** Una medición sobre un solo tamaño no permite hablar de complejidad: hay que ver cómo cambia el tiempo cuando crece el volumen. Hasta el tamaño real se usan muestras del conjunto; sobre él se repiten filas al azar (con reemplazo) solo para observar la tendencia.

In [31]:
# Cómo crece el costo con el tamaño: esto es lo que revela la complejidad
mediciones = []
for n in [1_000, 5_000, len(serie), 50_000, 100_000]:
    # Sobre el tamaño real se repiten filas al azar, solo para ver la tendencia
    muestra = serie.sample(n=n, replace=n > len(serie), random_state=SEMILLA)
    t_b = min(timeit.repeat(lambda: escalar_con_bucle(muestra), number=10, repeat=5)) / 10
    t_v = min(timeit.repeat(lambda: escalar_vectorizado(muestra), number=10, repeat=5)) / 10
    _, _, m_b = medir(escalar_con_bucle, muestra)
    _, _, m_v = medir(escalar_vectorizado, muestra)
    mediciones.append({"filas": n, "bucle_ms": round(t_b*1000, 3),
                       "vectorizada_ms": round(t_v*1000, 3), "razón": round(t_b/t_v, 1),
                       "bucle_KB": round(m_b*1024, 1), "vectorizada_KB": round(m_v*1024, 1)})

print(pd.DataFrame(mediciones).to_string(index=False))


 filas  bucle_ms  vectorizada_ms  razón  bucle_KB  vectorizada_KB
  1000     0.169           0.065    2.6      88.1            18.4
  5000     0.759           0.064   11.9     440.6            80.9
 14758     2.165           0.075   28.7    1299.7           233.5
 50000     7.131           0.101   70.8    4437.2           784.0
100000    14.728           0.139  105.7    8789.3          1565.3


**Cómo se lee la tabla.**

- **Tiempo del bucle.** Crece en proporción a las filas: de 1.000 a 100.000 filas (100 veces más) pasa de 0,18 a 15,2 ms, unas 86 veces más. Es complejidad temporal **lineal, O(n)**: cada valor se procesa una vez, en Python.
- **Tiempo de la vectorizada.** Casi no cambia: de 0,06 a 0,14 ms en el mismo rango. También es O(n), porque NumPy debe procesar cada valor, pero lo hace en código compilado, y a estos tamaños domina un costo fijo de preparación que no depende del número de filas. Por eso la razón entre ambas crece de 3 a 110 veces y no se estabiliza dentro del rango medido.
- **Memoria.** Las dos crecen en proporción a las filas (complejidad espacial **O(n)**), pero la vectorizada usa entre 4,7 y 5,7 veces menos (cerca de 5,6 desde el tamaño real), porque guarda cada valor en 8 bytes, mientras que el bucle crea un objeto de Python por valor.

**Tres reglas que se cumplieron para que la medición valga**

1. **Se repitió y se conservó el tiempo menor.** `timeit.repeat` mide 5 veces y se toma el mínimo: los valores altos suelen reflejar interrupciones del sistema operativo, no el costo del código.
2. **Se midió sobre el tamaño real**: los 14.758 alojamientos completos que procesa el pipeline, además de tamaños menores y mayores para ver la tendencia.
3. **Se comprobó que las versiones dan lo mismo**, con `assert`, antes de comparar tiempos.

**La decisión que tomó la medición.** `EscaladorRobusto.aplicar()` usa la forma vectorizada: es más rápida y además usa menos memoria, así que no hay intercambio que evaluar. Sin embargo, el escalado tarda menos de 1 ms sobre el conjunto real. Para justificar una optimización hay que medir dónde se va realmente el tiempo del pipeline, y eso hacemos a continuación.

## ¿Cuál es el paso más costoso del pipeline?

El escalado es una de las operaciones más baratas del pipeline: menos de 1 ms sobre el conjunto real. Para que una optimización se justifique, hay que saber dónde se va realmente el tiempo. Por eso medimos cada uno de los 24 pasos sobre los datos reales, en el mismo orden en que los ejecuta el pipeline, con `timeit` (5 repeticiones, se conserva el mínimo).

In [32]:
# Tiempo de cada paso del pipeline real, para encontrar el más costoso
tiempos = []
intermedio = datos.copy()
for paso in construir_pipeline().pasos_ejecutados():
    paso.ajustar(intermedio)
    t = min(timeit.repeat(lambda: paso.transformar(intermedio), number=3, repeat=5)) / 3
    intermedio = paso.transformar(intermedio)
    tiempos.append({"paso": paso.nombre, "ms": round(t * 1000, 2)})

tabla_pasos = pd.DataFrame(tiempos).sort_values("ms", ascending=False)
print(f"Tiempo total: {tabla_pasos['ms'].sum():.0f} ms · ExtractorComodidades: "
      f"{tabla_pasos['ms'].iloc[0] / tabla_pasos['ms'].sum():.0%} del total\n")
print(tabla_pasos.head(5).to_string())

Tiempo total: 603 ms · ExtractorComodidades: 85% del total

                                          paso      ms
14             ExtractorComodidades(amenities)  513.04
1     FiltroFilas(solo alojamientos completos)   26.35
16  CodificadorNominal(neighbourhood_cleansed)   19.69
0                       ConversorPrecio(price)   19.37
2               SelectorColumnas(11 variables)    5.90


**Cómo se lee.** `ExtractorComodidades` es, por lejos, el paso más costoso: concentra el 85 % del tiempo total del pipeline (unos 511.76 de 601 ms), y el segundo paso más lento (el filtro de alojamientos completos, 22.75 ms) tarda unas 20 veces menos. Por cada uno de los 14.758 alojamientos hace tres cosas: convierte con `json` un texto de unos 570 caracteres en una lista de Python (32 comodidades en promedio), une esa lista en un solo texto y busca en él ocho expresiones regulares. Los demás pasos son operaciones vectorizadas sobre columnas numéricas o categóricas.

Este es el paso donde una optimización tendría efecto real. En la celda siguiente comparamos dos implementaciones.

In [33]:
import json
import re

# Datos reales tal como llegan al extractor: se aplican todos los pasos anteriores a él
pasos = construir_pipeline().pasos_ejecutados()
indice = next(i for i, p in enumerate(pasos) if isinstance(p, ExtractorComodidades))
antes_del_extractor = datos.copy()
for paso in pasos[:indice]:
    antes_del_extractor = paso.ajustar_transformar(antes_del_extractor)
amenities = antes_del_extractor["amenities"]


def comodidades_con_json(columna):
    """Versión A (la actual): convierte cada texto en lista con json, la une y busca los patrones."""
    listas = columna.apply(json.loads)
    texto = listas.apply(lambda lista: "|".join(lista).lower())
    salida = pd.DataFrame({"n_comodidades": listas.str.len()})
    for nueva, patron in COMODIDADES.items():
        salida[nueva] = texto.str.contains(patron, regex=True).astype(int)
    return salida


def comodidades_sin_json(columna):
    """Versión B: busca los patrones directamente en el texto, sin crear listas de Python."""
    texto = columna.str.lower()
    # Cada separador '", "' divide dos elementos; la lista vacía "[]" tiene cero
    conteo = (texto.str.count(re.escape('", "')) + 1).where(texto != "[]", 0)
    salida = pd.DataFrame({"n_comodidades": conteo})
    for nueva, patron in COMODIDADES.items():
        salida[nueva] = texto.str.contains(patron, regex=True).astype(int)
    return salida


# Antes de comparar: las dos deben dar EXACTAMENTE lo mismo
pd.testing.assert_frame_equal(comodidades_con_json(amenities), comodidades_sin_json(amenities))
print(f"Las dos versiones dan el mismo resultado sobre los {len(amenities)} alojamientos.\n")

t_a = min(timeit.repeat(lambda: comodidades_con_json(amenities), number=3, repeat=5)) / 3
t_b = min(timeit.repeat(lambda: comodidades_sin_json(amenities), number=3, repeat=5)) / 3
_, _, mem_a = medir(comodidades_con_json, amenities)
_, _, mem_b = medir(comodidades_sin_json, amenities)

print(f"A · con json : {t_a*1000:.1f} ms   {mem_a:.1f} MB")
print(f"B · sin json : {t_b*1000:.1f} ms   {mem_b:.1f} MB")
print(f"\nB es {t_a / t_b:.2f} veces más rápida y usa {mem_b / mem_a:.2f} veces la memoria de A.")

Las dos versiones dan el mismo resultado sobre los 14758 alojamientos.

A · con json : 503.2 ms   42.6 MB
B · sin json : 534.6 ms   9.9 MB

B es 0.94 veces más rápida y usa 0.23 veces la memoria de A.


**Cómo se lee.** El `assert_frame_equal` confirma que las dos versiones producen el mismo conteo y los mismos ocho indicadores en los 14.758 alojamientos. La diferencia está en cómo llegan a ese resultado:

- **A (con json)** convierte el texto de cada alojamiento en una lista de Python, la une en un solo texto y busca allí los ocho patrones.
- **B (sin json)** busca los patrones directamente en el texto original y cuenta las comodidades por sus separadores.

**Las mediciones muestran un intercambio entre tiempo y memoria:**

- **Tiempo: prácticamente empatan** (A 517 ms, B 542 ms; B es cerca de un 5 % más lenta). B se ahorra la conversión con `json` y la unión de cada lista, pero recorre el texto original, que es más largo porque conserva comillas y separadores, y además necesita una búsqueda extra para contar los separadores. Las ocho búsquedas de patrones, que las dos versiones hacen, siguen siendo la parte más cara. Ambas son O(n × L), donde n es el número de alojamientos y L el largo del texto.
- **Memoria: B usa cerca de una cuarta parte** (9,9 MB contra 42,6 MB). A crea una lista por alojamiento con unas 32 comodidades: cerca de 475 mil objetos de Python en total. B no crea esas estructuras y trabaja sobre una sola copia en minúsculas del texto.

**Decisión.** Mantenemos A en `ExtractorComodidades`. Con 14.758 alojamientos, sus 43 MB no son una restricción para un computador actual, el tiempo es prácticamente el mismo, y A interpreta el JSON fila por fila, lo que permite detectar filas mal formadas: agregamos el manejo de ese error, que ahora lanza `ProyectoError` indicando la fila. B sería la opción preferible si la memoria pasara a ser la limitante, por ejemplo al procesar varias capturas de Inside Airbnb al mismo tiempo.

---
# 11. Patrones de diseño

Un patrón es una solución conocida a un problema que se repite. Lo importante:
**ustedes ya usaron varios en la Fase 2 sin saber su nombre.**

## Strategy: varias formas de hacer lo mismo

En la Fase 2 comparamos imputar `beds` con la mediana, con la media o por grupo (por ejemplo, según el tipo de habitación o comuna). Si cada forma es un bloque distinto, cambiar de una a otra obliga a reescribir. Strategy separa el **qué** del **cómo**.

In [34]:
class EstrategiaImputacion:
    """Contrato común de todas las estrategias."""
    etiqueta = "sin definir"

    def calcular(self, df, columna):
        raise NotImplementedError


class PorMediana(EstrategiaImputacion):
    etiqueta = "mediana"

    def calcular(self, df, columna):
        return float(df[columna].median())


class PorMedia(EstrategiaImputacion):
    etiqueta = "media"

    def calcular(self, df, columna):
        return float(df[columna].mean())


class PorMedianaDeGrupo(EstrategiaImputacion):
    etiqueta = "mediana por grupo"

    def __init__(self, columna_grupo):
        self.columna_grupo = columna_grupo

    def calcular(self, df, columna):
        return df.groupby(self.columna_grupo)[columna].median().to_dict()


class ImputadorFlexible(Transformador):
    """No sabe imputar: sabe cuándo. El cómo lo aporta la estrategia."""

    def __init__(self, columna, estrategia=None):
        super().__init__(columna)          # llama al constructor de la clase base
        self.estrategia = estrategia or PorMediana()

    def aprender(self, df):
        return {"valor": self.estrategia.calcular(df, self.columna),
                "estrategia": self.estrategia.etiqueta}

    def aplicar(self, df):
        valor = self._parametros["valor"]
        if isinstance(valor, dict):                       # mediana por grupo
            relleno = df[self.estrategia.columna_grupo].map(valor)
            df[self.columna] = df[self.columna].fillna(relleno)
            df[self.columna] = df[self.columna].fillna(df[self.columna].median())
        else:
            df[self.columna] = df[self.columna].fillna(valor)
        return df


# La comparación de estrategias que pide la rúbrica, en un bucle adaptada a Airbnb
desv_original = datos["beds"].std()
comparacion = []

# Agrupamos por 'room_type' (tipo de alojamiento) para la estrategia por grupo
for estrategia in [PorMediana(), PorMedia(), PorMedianaDeGrupo("room_type")]:
    salida = ImputadorFlexible("beds", estrategia).ajustar_transformar(datos)
    comparacion.append({
        "estrategia": estrategia.etiqueta,
        "desv_antes": round(desv_original, 4),
        "desv_despues": round(salida["beds"].std(), 4),
        "cambio_pct": round((salida["beds"].std() - desv_original) / desv_original * 100, 2),
    })

pd.DataFrame(comparacion)

,estrategia,desv_antes,desv_despues,cambio_pct
0,mediana,1.842,1.8321,-0.54
1,media,1.842,1.8320,-0.54
2,mediana por grupo,1.842,1.8328,-0.49


**Nota sobre estos números.** En el archivo completo, `beds` tiene 203 faltantes de 18.904 filas (cerca de un 1 %). Con tan pocos huecos, las tres estrategias cambian la desviación estándar en menos de un 1 % y son prácticamente equivalentes. Es la misma conclusión de la comparación de imputaciones de la Fase 2: cuando las estrategias no se distinguen, se declara y se elige la más simple, que es la mediana, coherente además con la asimetría de las variables del estudio.

Esa tabla es la comparación que el criterio de preprocesamiento pide justificar. Toda
imputación por un valor central reduce la dispersión; lo que se compara es **cuánto**, y
esa cifra es el argumento para elegir.

## Factory: decidir qué construir

La transformación que corresponde depende del rol de la variable. Factory concentra esa
decisión en un solo lugar, en vez de repetirla en cada cuaderno.

In [35]:
def crear_transformador(rol, columna, **parametros):
    """Devuelve el transformador que corresponde al rol analítico."""
    rol = rol.strip().lower()

    if rol == "continua":
        return ImputadorFlexible(columna, parametros.get("estrategia"))
    if rol == "nominal":
        return CodificadorNominal(columna)
    if rol == "escalar":
        return EscaladorRobusto(columna)
    if rol == "identificador":
        return EliminadorColumnas(columna)

    raise ValueError(
        f"Rol desconocido: '{rol}'. "
        "Roles válidos: continua, nominal, escalar, identificador."
    )


# El diccionario de variables de la Fase 1 adaptado al proyecto de Airbnb
diccionario = [
    {"variable": "id", "rol": "identificador"},
    {"variable": "beds", "rol": "continua"},
    {"variable": "room_type", "rol": "nominal"},
    {"variable": "neighbourhood_cleansed", "rol": "nominal"},
    {"variable": "minimum_nights", "rol": "escalar"},
]

pipeline_auto = Pipeline([crear_transformador(v["rol"], v["variable"]) for v in diccionario])
salida = pipeline_auto.ajustar(datos).transformar(datos)

print(pipeline_auto.resumen().to_string(index=False))
print("\nResultado:", salida.shape)

try:
    crear_transformador("geografica", "latitude")
except ValueError as error:
    print("\nValueError:", error)

 orden                                       paso              clase
     1                     EliminadorColumnas(id) EliminadorColumnas
     2                    ImputadorFlexible(beds)  ImputadorFlexible
     3              CodificadorNominal(room_type) CodificadorNominal
     4 CodificadorNominal(neighbourhood_cleansed) CodificadorNominal
     5           EscaladorRobusto(minimum_nights)   EscaladorRobusto

Resultado: (18904, 123)

ValueError: Rol desconocido: 'geografica'. Roles válidos: continua, nominal, escalar, identificador.


## Observer: registrar sin estorbar

La bitácora de decisiones suele escribirse a mano después de ejecutar, con el riesgo de
quedar desactualizada. Con Observer, el pipeline **avisa** cada vez que termina un paso
y quien quiera registrar se suscribe.

In [36]:
class Bitacora:
    """Observador que acumula lo que ocurre y lo entrega como tabla."""

    def __init__(self):
        self.registros = []

    def notificar(self, paso, filas, columnas, segundos, parametros):
        self.registros.append({"paso": paso, "filas": filas, "columnas": columnas,
                               "segundos": round(segundos, 5), "parametros": parametros})

    def a_dataframe(self):
        return pd.DataFrame(self.registros)


class ReporteConsola:
    """Otro observador: imprime mientras ocurre."""

    def notificar(self, paso, filas, columnas, segundos, parametros):
        print(f"  {paso:<34} {filas:>6} filas, {columnas:>3} col  [{segundos:.4f}s]")


class PipelineObservable(Pipeline):
    """Hereda todo de Pipeline y agrega la capacidad de ser observado."""

    def __init__(self, pasos=None):
        super().__init__(pasos)
        self._observadores = []

    def suscribir(self, observador):
        self._observadores.append(observador)
        return self

    def transformar(self, df):
        if not self._ajustado:
            raise RuntimeError("Pipeline: hay que ajustar antes de transformar.")
        resultado = df.copy()
        for paso in self._pasos:
            inicio = time.perf_counter()
            resultado = paso.transformar(resultado)
            transcurrido = time.perf_counter() - inicio
            for observador in self._observadores:
                observador.notificar(paso.nombre, len(resultado), resultado.shape[1],
                                     transcurrido, paso.parametros)
        return resultado


# Instanciamos la bitácora y armamos el pipeline observable adaptado a Airbnb
bitacora = Bitacora()
pipe = PipelineObservable([
    EliminadorColumnas("id"),
    ImputadorFlexible("beds"),
    CodificadorNominal("room_type"),
    EscaladorRobusto("minimum_nights"),
])
pipe.suscribir(bitacora).suscribir(ReporteConsola())

print("Ejecución del pipeline:")
salida = pipe.ajustar(datos).transformar(datos)

print("\nBitácora producida por la propia ejecución:")
bitacora.a_dataframe()[["paso", "filas", "columnas", "segundos"]]

Ejecución del pipeline:
  EliminadorColumnas(id)              18904 filas,  89 col  [0.0075s]
  ImputadorFlexible(beds)             18904 filas,  89 col  [0.0095s]
  CodificadorNominal(room_type)       18904 filas,  92 col  [0.0137s]
  EscaladorRobusto(minimum_nights)    18904 filas,  92 col  [0.0110s]

Bitácora producida por la propia ejecución:


,paso,filas,columnas,segundos
0,EliminadorColumnas(id),18904,89,0.00751
1,ImputadorFlexible(beds),18904,89,0.00951
2,CodificadorNominal(room_type),18904,92,0.01370
3,EscaladorRobusto(minimum_nights),18904,92,0.01099


**Fíjense en lo que acaba de pasar.** La bitácora no se escribió a mano: se genera cada
vez que el pipeline corre, así que no puede quedar desactualizada.

## Singleton: una sola configuración

Rutas, semilla y umbrales se necesitan en varios módulos. Si cada uno construye su
propia configuración, dos partes del pipeline pueden usar semillas distintas sin que
nadie lo note.

**La advertencia:** Singleton introduce estado global, que complica las pruebas. Úsenlo
solo para configuración y pasen el resto de los parámetros explícitamente.

In [37]:
class Configuracion:
    """Una sola instancia compartida por todo el proyecto."""

    _instancia = None

    def __new__(cls, *args, **kwargs):
        if cls._instancia is None:
            cls._instancia = super().__new__(cls)
            cls._instancia._iniciada = False
        return cls._instancia

    def __init__(self, semilla=SEMILLA):
        if self._iniciada:               # evita reiniciar una instancia existente
            return
        self.semilla = semilla
        self.parametros = {}
        self._iniciada = True

    def definir(self, clave, valor):
        self.parametros[clave] = valor
        return self


# Instanciación y prueba del Singleton adaptado a Airbnb
a = Configuracion(semilla=SEMILLA)
b = Configuracion(semilla=99)            # se ignora: la instancia ya existe

# Definimos parámetros propios de tu pipeline de alojamientos
a.definir("umbral_nulos", 0.6).definir("test_size", 0.2).definir("objetivos", OBJETIVOS)

print("¿Son el mismo objeto?", a is b)
print("Semilla vista desde b:", b.semilla)
print("Parámetros desde b   :", b.parametros)

¿Son el mismo objeto? True
Semilla vista desde b: 42
Parámetros desde b   : {'umbral_nulos': 0.6, 'test_size': 0.2, 'objetivos': ['price', 'estimated_occupancy_l365d']}


## Resumen: qué patrón usar y cuándo

| Patrón | El problema que resuelve | Dónde ya lo hicieron en la Fase 2 |
|---|---|---|
| **Strategy** | Varias formas de hacer lo mismo | Comparar imputaciones de `bmi` |
| **Factory** | Elegir qué construir según una condición | Decidir la transformación según el rol |
| **Observer** | Registrar sin que el ejecutor sepa quién anota | La bitácora de decisiones |
| **Singleton** | Una sola configuración compartida | La semilla y las rutas del proyecto |

**No usen los cuatro.** Usen el que su proyecto necesita y expliquen por qué. Un patrón
aplicado donde no hace falta agrega complejidad sin beneficio, y eso también se evalúa.

---
---
# 12. El pipeline construido desde la configuración

El pipeline de la sección 6.1 no tiene ningún nombre de columna escrito a mano: todo sale de la celda de configuración. Es Factory aplicado al proyecto completo: la configuración deja de ser documentación y pasa a gobernar el código.

In [38]:
pipeline_config = construir_pipeline()
print(f"Pipeline construido con {len(pipeline_config)} pasos desde la configuración:\n")
print(pipeline_config.resumen().to_string(index=False))

Pipeline construido con 24 pasos desde la configuración:

 orden                                       paso                clase
     1                     ConversorPrecio(price)      ConversorPrecio
     2   FiltroFilas(solo alojamientos completos)          FiltroFilas
     3             SelectorColumnas(11 variables)     SelectorColumnas
     4                   Acotador(maximum_nights)             Acotador
     5       FiltroFilas(objetivos sin faltantes)          FiltroFilas
     6                    MarcadorNulos(bedrooms)        MarcadorNulos
     7             ImputadorMediana(accommodates)     ImputadorMediana
     8                 ImputadorMediana(latitude)     ImputadorMediana
     9                 ImputadorMediana(bedrooms)     ImputadorMediana
    10                     ImputadorMediana(beds)     ImputadorMediana
    11           ImputadorMediana(minimum_nights)     ImputadorMediana
    12           ImputadorMediana(maximum_nights)     ImputadorMediana
    13          Fil

In [39]:
# Se ajusta con entrenamiento y se aplica a prueba, sin fuga de datos
pipeline_config.ajustar(entrenamiento)
entrenamiento_listo = pipeline_config.transformar(entrenamiento)
prueba_lista = pipeline_config.transformar(prueba)

print("Entrenamiento:", entrenamiento.shape, "->", entrenamiento_listo.shape)
print("Prueba       :", prueba.shape, "->", prueba_lista.shape)

# Las dos salidas deben tener exactamente las mismas columnas y en el mismo orden
assert list(entrenamiento_listo.columns) == list(prueba_lista.columns), \
    "Entrenamiento y prueba quedaron con columnas distintas"
print("\nAmbos conjuntos tienen las mismas columnas, en el mismo orden.")

Entrenamiento: (15123, 90) -> (11788, 50)
Prueba       : (3781, 90) -> (2970, 50)

Ambos conjuntos tienen las mismas columnas, en el mismo orden.


**Esa última comprobación es más importante de lo que parece.** Si el codificador
aprendiera el vocabulario de cada conjunto por separado, entrenamiento y prueba
quedarían con columnas distintas y cualquier modelo posterior fallaría. Como el
vocabulario se aprende una sola vez en el ajuste, eso no puede pasar.

## Guardar el resultado y su configuración

El conjunto procesado no sirve de mucho sin el registro de cómo se produjo. Guardar
ambos juntos es lo que hace el trabajo trazable.

In [40]:
def guardar_resultado(df, ruta_salida, pipeline, carpeta=DIR_PROCESADO):
    """Escribe el conjunto procesado y un registro de los parámetros aprendidos."""
    os.makedirs(carpeta, exist_ok=True)
    destino = os.path.join(carpeta, ruta_salida)
    df.to_csv(destino, index=False)

    registro = pd.DataFrame([
        {"orden": i, "paso": p.nombre, "parametros": str(p.parametros)}
        for i, p in enumerate(pipeline.pasos_ejecutados(), start=1)
    ])
    registro.to_csv(destino.replace(".csv", "_parametros.csv"), index=False)

    print(f"Conjunto guardado   : {destino} ({df.shape[0]} filas x {df.shape[1]} col)")
    print(f"Parámetros guardados: {destino.replace('.csv', '_parametros.csv')}")
    return registro


registro = guardar_resultado(entrenamiento_listo, "entrenamiento_procesado.csv",
                             pipeline_config)
registro.head()

Conjunto guardado   : c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\processed\entrenamiento_procesado.csv (11788 filas x 50 col)
Parámetros guardados: c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\processed\entrenamiento_procesado_parametros.csv


,orden,paso,parametros
0,1,ConversorPrecio(price),"{'regla': ""quitar '$' y ',' y convertir a núme..."
1,2,FiltroFilas(solo alojamientos completos),{'regla': 'solo alojamientos completos'}
2,3,SelectorColumnas(11 variables),{'conservadas': 11}
3,4,Acotador(maximum_nights),"{'maximo': 1125, 'valores_sobre_el_maximo': 2}"
4,5,FiltroFilas(objetivos sin faltantes),{'regla': 'objetivos sin faltantes'}


---
# 13. El resultado: cómo queda el código y cómo quedan los datos

Esta sección cierra el recorrido. Muestra dos cosas: **cómo se ve el proyecto después de
aplicar POO** y **qué conjunto queda listo para la Fase 4**.

## 13.1 El código, antes y después

**Antes (Fase 2).** El pipeline era una secuencia de celdas con funciones. Funcionaba, pero el orden estaba implícito y cada paso calculaba sus parámetros sobre el conjunto completo:

```python
df_crudo["price"] = pd.to_numeric(df_crudo["price"].str.replace("$", "")...)
df = df_crudo[df_crudo["room_type"] == "Entire home/apt"][variables_estudio]
df["maximum_nights"] = df["maximum_nights"].clip(upper=1125)
df = df.dropna(subset=OBJETIVOS)
for col in columnas_imputar:
    df = imputar_nulos_numericos(df, col, estrategia="mediana")
...
df = codificar_one_hot(df, "neighbourhood_cleansed", nombres_comunas)
df, scaler = escalar_caracteristicas(df, columnas_a_escalar)
```

**Después (Fase 3).** El mismo trabajo, en cuatro líneas:

```python
pipeline = construir_pipeline()
pipeline.ajustar(entrenamiento)
entrenamiento_listo = pipeline.transformar(entrenamiento)
prueba_lista = pipeline.transformar(prueba)
```

Lo que cambió no es el resultado (la sección 8 lo comprueba): ahora cada paso es una pieza con nombre, que se puede probar, reemplazar y reutilizar sin tocar las demás, y que aprende sus parámetros solo del conjunto de entrenamiento.

## 13.2 La arquitectura, generada desde el propio código

En vez de escribir a mano la tabla de arquitectura para el informe, se puede producir
leyendo las clases que existen. Así no queda desactualizada.

In [41]:
def documentar_arquitectura():
    """Genera la tabla de arquitectura a partir de las clases definidas.

    __subclasses__() devuelve las clases que heredan de Transformador. Es la
    misma información que Python usa para resolver la herencia, así que la
    tabla refleja el código real y no lo que creemos que hay.
    """
    filas = [{
        "componente": "Transformador",
        "rol": "clase base",
        "responsabilidad": "Define el contrato y controla el estado de cada paso",
        "archivo": "src/transformadores.py",
    }]

    for clase in Transformador.__subclasses__():
        resumen = (clase.__doc__ or "sin documentar").strip().split("\n")[0]
        filas.append({
            "componente": clase.__name__,
            "rol": "clase hija",
            "responsabilidad": resumen,
            "archivo": (f"src/{clase.__module__.split('.')[-1]}.py"
                        if clase.__module__.startswith("src.") else "cuaderno (ejemplo)"),
        })

    filas += [
        {"componente": "Pipeline", "rol": "orquestador",
         "responsabilidad": "Encadena los pasos y controla el orden de ajuste",
         "archivo": "src/pipeline.py"},
        {"componente": "PipelineObservable", "rol": "orquestador",
         "responsabilidad": "Pipeline que además notifica cada paso a sus observadores",
         "archivo": "cuaderno (ejemplo del patrón Observer)"},
        {"componente": "Bitacora", "rol": "observador",
         "responsabilidad": "Registra lo ocurrido en cada paso de la ejecución",
         "archivo": "cuaderno (ejemplo del patrón Observer)"},
        {"componente": "construir_pipeline", "rol": "fábrica",
         "responsabilidad": "Arma el pipeline desde la celda de configuración",
         "archivo": "src/pipeline.py"},
        {"componente": "cargar", "rol": "carga",
         "responsabilidad": "Lee el archivo crudo y verifica su esquema",
         "archivo": "src/carga.py"},
        {"componente": "medir", "rol": "utilidad",
         "responsabilidad": "Mide tiempo y memoria de cualquier función",
         "archivo": "src/medicion.py"},
    ]
    return pd.DataFrame(filas)


arquitectura = documentar_arquitectura()
print(arquitectura.to_string(index=False))

          componente         rol                                                            responsabilidad                                archivo
       Transformador  clase base                       Define el contrato y controla el estado de cada paso                 src/transformadores.py
    ImputadorMediana  clase hija                   Rellena los nulos con la mediana aprendida en el ajuste.                 src/transformadores.py
  CodificadorNominal  clase hija             Convierte una columna de categorías en columnas 0/1 (One-Hot).                 src/transformadores.py
    EscaladorRobusto  clase hija Centra en la mediana y divide por el rango intercuartílico (RobustScaler).                 src/transformadores.py
  EliminadorColumnas  clase hija          Quita columnas que no aportan al análisis, como el identificador.                 src/transformadores.py
     ConversorPrecio  clase hija                      Convierte el precio de texto ("$55,106.00") a número.           

**Esa tabla es el apartado de documentación de arquitectura del informe.** Cópienla,
agreguen dos o tres párrafos explicando **por qué** esa división, y el criterio queda
cubierto.

La estructura de archivos que sugiere la última columna es la que corresponde llevar al
repositorio. Los módulos van en la carpeta `src/` de la raíz, compartida por todas las
fases; en la carpeta de la fase queda solo el cuaderno:

```
proyecto-grupoX-mcdi500/
├── data/processed/                  lo que se entrega a la Fase 4
├── src/
│   ├── carga.py                     lectura y verificación del archivo
│   ├── transformadores.py           la clase base y sus hijas
│   ├── pipeline.py                  Pipeline
│   └── medicion.py                  medir()
└── F3/
    └── S2_F3_NucleoAlgoritmico_Eficiencia_POO.ipynb   este cuaderno, ejecutado
```

`PipelineObservable`, `Bitacora` y la fábrica son extensiones opcionales: si su proyecto
las usa, van en `src/` con el mismo criterio (por ejemplo `src/observadores.py`).


## 13.3 El conjunto procesado: qué queda listo

Acá se ejecuta el pipeline completo y se revisa el resultado con el detalle que la
Fase 4 va a necesitar.

In [42]:
# Pipeline definitivo, construido desde la configuración
pipeline_final = construir_pipeline()
pipeline_final.ajustar(entrenamiento)

entrenamiento_final = pipeline_final.transformar(entrenamiento)
prueba_final = pipeline_final.transformar(prueba)

print("CONJUNTO PROCESADO")
print("=" * 58)
print(f"Entrenamiento : {entrenamiento_final.shape[0]:>6} filas x {entrenamiento_final.shape[1]:>3} columnas")
print(f"Prueba        : {prueba_final.shape[0]:>6} filas x {prueba_final.shape[1]:>3} columnas")
print(f"Partida desde : {datos.shape[0]:>6} filas x {datos.shape[1]:>3} columnas")
print(f"\nColumnas finales: {entrenamiento_final.shape[1]} (las mismas 50 de la Fase 2)")
entrenamiento_final.head(3)

CONJUNTO PROCESADO
Entrenamiento :  11788 filas x  50 columnas
Prueba        :   2970 filas x  50 columnas
Partida desde :  18904 filas x  90 columnas

Columnas finales: 50 (las mismas 50 de la Fase 2)


,price,estimated_occupancy_l365d,accommodates,bedrooms,beds,latitude,minimum_nights,maximum_nights,bedrooms_imputado,n_comodidades,...,comuna_Pudahuel,comuna_Quilicura,comuna_Quinta_Normal,comuna_Recoleta,comuna_Renca,comuna_San_Joaquín,comuna_San_Miguel,comuna_Santiago,comuna_Vitacura,comuna_Ñuñoa
2388,84947.0,36,0.5,1.0,0.5,-0.512463,0.0,1.0,0,0.454545,...,0,0,0,0,0,0,0,0,0,1
9680,90000.0,0,-0.5,0.0,-0.5,1.139384,3.0,0.0,0,-1.227273,...,0,0,0,0,0,0,0,0,1,0
4489,43500.0,0,-0.5,0.0,-0.5,-0.451679,-1.0,1.0,0,0.727273,...,0,0,0,0,0,0,0,1,0,0


In [43]:
# Esquema final: qué tipo tiene cada columna y de dónde salió
def describir_esquema(df, originales):
    """Clasifica cada columna del resultado según su origen."""
    filas = []
    for columna in df.columns:
        if columna in originales:
            origen = "original"
        elif "_" in columna and columna.split("_")[0] in originales:
            origen = f"derivada de {columna.split('_')[0]}"
        else:
            origen = "derivada"
        filas.append({
            "columna": columna,
            "tipo": str(df[columna].dtype),
            "origen": origen,
            "nulos": int(df[columna].isna().sum()),
        })
    return pd.DataFrame(filas)


esquema = describir_esquema(entrenamiento_final, set(datos.columns))
print(f"Total de columnas: {len(esquema)}")
print(f"Originales conservadas: {(esquema['origen'] == 'original').sum()}")
print(f"Derivadas de la codificación: {(esquema['origen'] != 'original').sum()}")
print(f"Columnas con nulos: {(esquema['nulos'] > 0).sum()}")
print()
esquema.head(12)

Total de columnas: 50
Originales conservadas: 8
Derivadas de la codificación: 42
Columnas con nulos: 0



,columna,tipo,origen,nulos
0,price,float64,original,0
1,estimated_occupancy_l365d,int64,original,0
2,accommodates,float64,original,0
3,bedrooms,float64,original,0
4,beds,float64,original,0
5,latitude,float64,original,0
6,minimum_nights,float64,original,0
7,maximum_nights,float64,original,0
8,bedrooms_imputado,int64,derivada de bedrooms,0
9,n_comodidades,float64,derivada,0


## 13.3 bis Del cuaderno a los módulos

El criterio de diseño estructurado no se cumple con clases escritas dentro del cuaderno: el código reutilizable vive en `src/` y el cuaderno lo **importa** en cada sección donde lo usa.

| Archivo | Qué contiene | Dónde se importa |
|---|---|---|
| `src/carga.py` | `leer_archivo`, `verificar_esquema` y `cargar` | Carga del conjunto |
| `src/transformadores.py` | `Transformador` y las 10 clases de pasos que heredan de ella | Sección 5 |
| `src/pipeline.py` | `Pipeline` y `construir_pipeline(config)` | Secciones 6 y 6.1 |
| `src/medicion.py` | `medir`: tiempo y memoria pico | Sección 10 |

Los módulos no guardan rutas ni nombres de columna: el cuaderno se los entrega (`RUTA_DATOS`, `COLUMNAS_ESPERADAS`, `CONFIG`), así la configuración vive en un solo lugar. Las clases que siguen escritas en el cuaderno son ejemplos didácticos (secciones 2 a 4, 7 y 11).

La celda siguiente comprueba de dónde viene cada componente que usa el pipeline. La equivalencia con la Fase 2 ya quedó demostrada en la sección 8, que usa exactamente estos módulos.

In [44]:
import inspect
from pathlib import Path

componentes = [cargar, Transformador, ImputadorMediana, CodificadorNominal,
               EscaladorRobusto, EliminadorColumnas, ConversorPrecio, FiltroFilas,
               SelectorColumnas, Acotador, MarcadorNulos, ExtractorComodidades,
               Pipeline, construir_pipeline_desde_config, medir]

origen = pd.DataFrame([{
    "componente": c.__name__,
    "módulo": c.__module__,
    "archivo": "src/" + Path(inspect.getfile(c)).name,
} for c in componentes])

# Si algún componente se hubiera redefinido en el cuaderno, su módulo sería __main__
assert origen["módulo"].str.startswith("src.").all(), "Hay componentes definidos en el cuaderno"

print(origen.to_string(index=False))
print(f"\nLos {len(origen)} componentes del pipeline se importan desde src/.")

          componente              módulo                archivo
              cargar           src.carga           src/carga.py
       Transformador src.transformadores src/transformadores.py
    ImputadorMediana src.transformadores src/transformadores.py
  CodificadorNominal src.transformadores src/transformadores.py
    EscaladorRobusto src.transformadores src/transformadores.py
  EliminadorColumnas src.transformadores src/transformadores.py
     ConversorPrecio src.transformadores src/transformadores.py
         FiltroFilas src.transformadores src/transformadores.py
    SelectorColumnas src.transformadores src/transformadores.py
            Acotador src.transformadores src/transformadores.py
       MarcadorNulos src.transformadores src/transformadores.py
ExtractorComodidades src.transformadores src/transformadores.py
            Pipeline        src.pipeline        src/pipeline.py
  construir_pipeline        src.pipeline        src/pipeline.py
               medir        src.medicion

## 13.4 Verificación de entrega

Antes de pasar a la Fase 4 conviene comprobar cinco condiciones. Si alguna falla, el
problema es de esta fase y corregirlo acá cuesta mucho menos que descubrirlo después.

In [45]:
def verificar_entrega(entrenamiento_final, prueba_final, total_final, objetivos):
    """Cinco comprobaciones antes de entregar el conjunto a la Fase 4."""
    controles = []

    # 1. La partición no perdió ni duplicó filas: entrenamiento + prueba = total procesado
    obtenidas = len(entrenamiento_final) + len(prueba_final)
    controles.append(("Partición sin filas perdidas ni duplicadas",
                      obtenidas == len(total_final), f"{obtenidas} de {len(total_final)}"))

    # 2. Entrenamiento y prueba tienen el mismo esquema
    mismo_esquema = list(entrenamiento_final.columns) == list(prueba_final.columns)
    controles.append(("Mismo esquema en ambos conjuntos",
                      mismo_esquema, f"{entrenamiento_final.shape[1]} columnas"))

    # 3. No quedan nulos en ninguno de los dos conjuntos
    nulos = int(entrenamiento_final.isna().sum().sum() + prueba_final.isna().sum().sum())
    controles.append(("Sin valores nulos", nulos == 0, f"{nulos} nulos"))

    # 4. Los objetivos sobrevivieron completos
    objetivos_ok = all(o in entrenamiento_final.columns
                       and entrenamiento_final[o].isna().sum() == 0 for o in objetivos)
    controles.append(("Objetivos presentes y completos", objetivos_ok, ", ".join(objetivos)))

    # 5. Todo quedó numérico y sin identificador
    numerico = entrenamiento_final.select_dtypes(exclude="number").empty
    sin_id = COLUMNA_ID not in entrenamiento_final.columns
    controles.append(("Todo numérico y sin identificador", numerico and sin_id, COLUMNA_ID))

    tabla = pd.DataFrame(
        [{"control": c[0], "estado": "OK" if c[1] else "FALLA", "detalle": c[2]}
         for c in controles]
    )
    return tabla, all(c[1] for c in controles)


total_final = pipeline_final.transformar(datos)
controles, aprobado = verificar_entrega(entrenamiento_final, prueba_final,
                                        total_final, OBJETIVOS)
print(controles.to_string(index=False))
print("\n" + ("ENTREGA APROBADA: el conjunto puede pasar a la Fase 4."
               if aprobado else
               "ENTREGA RECHAZADA: revisar los controles en estado FALLA."))

                                   control estado                          detalle
Partición sin filas perdidas ni duplicadas     OK                   14758 de 14758
          Mismo esquema en ambos conjuntos     OK                      50 columnas
                         Sin valores nulos     OK                          0 nulos
           Objetivos presentes y completos     OK price, estimated_occupancy_l365d
         Todo numérico y sin identificador     OK                               id

ENTREGA APROBADA: el conjunto puede pasar a la Fase 4.


## 13.5 Lo que se entrega a la Fase 4

La Fase 4 no debe volver a limpiar ni transformar. Recibe cuatro artefactos y trabaja
sobre ellos.

In [46]:
def entregar_a_fase4(entrenamiento_final, prueba_final, pipeline, esquema,
                     carpeta=DIR_PROCESADO):
    """Escribe los cuatro artefactos que recibe la Fase 4."""
    os.makedirs(carpeta, exist_ok=True)
    entregables = {}

    # 1. Los conjuntos listos para analizar
    ruta_ent = os.path.join(carpeta, "entrenamiento_f3.csv")
    ruta_pru = os.path.join(carpeta, "prueba_f3.csv")
    entrenamiento_final.to_csv(ruta_ent, index=False)
    prueba_final.to_csv(ruta_pru, index=False)
    entregables["entrenamiento"] = ruta_ent
    entregables["prueba"] = ruta_pru

    # 2. El diccionario del conjunto resultante
    ruta_dic = os.path.join(carpeta, "diccionario_f3.csv")
    esquema.to_csv(ruta_dic, index=False)
    entregables["diccionario"] = ruta_dic

    # 3. Los parámetros aprendidos: es lo que permite reproducir el resultado
    ruta_par = os.path.join(carpeta, "parametros_f3.csv")
    pd.DataFrame([
        {"orden": i, "paso": p.nombre, "parametros": str(p.parametros)}
        for i, p in enumerate(pipeline.pasos_ejecutados(), start=1)
    ]).to_csv(ruta_par, index=False)
    entregables["parametros"] = ruta_par

    return entregables


entregables = entregar_a_fase4(entrenamiento_final, prueba_final,
                               pipeline_final, esquema)

print("ARTEFACTOS ENTREGADOS A LA FASE 4")
print("=" * 58)
for nombre, ruta in entregables.items():
    tamano = os.path.getsize(ruta) / 1024
    print(f"  {nombre:<16} {ruta:<40} {tamano:>7.1f} KB")

ARTEFACTOS ENTREGADOS A LA FASE 4
  entrenamiento    c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\processed\entrenamiento_f3.csv  1806.6 KB
  prueba           c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\processed\prueba_f3.csv   456.1 KB
  diccionario      c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\processed\diccionario_f3.csv     1.7 KB
  parametros       c:\Users\goku9\Desktop\grupo_8_magister_mcdi500\data\processed\parametros_f3.csv     2.3 KB


### El contrato entre fases

| Artefacto | Qué contiene | Para qué lo usa la Fase 4 |
|---|---|---|
| `entrenamiento_f3.csv` | El conjunto procesado de entrenamiento | Ajustar y explorar |
| `prueba_f3.csv` | El conjunto procesado de prueba | Evaluar, con el mismo esquema |
| `diccionario_f3.csv` | Columna, tipo, origen y nulos | Saber qué significa cada variable |
| `parametros_f3.csv` | Lo que aprendió cada paso | Reproducir el resultado o auditarlo |

**Lo que la Fase 4 NO debe hacer:** volver a imputar, volver a codificar ni volver a
escalar. Si lo hiciera sobre el conjunto ya transformado, escalaría dos veces y los
resultados dejarían de tener sentido.

**Lo que sí debe hacer:** leer estos archivos, explorar, visualizar y comunicar. Y si
necesita una transformación nueva, agregarla como un paso más del pipeline de la Fase 3,
no como código suelto en su propio cuaderno. Esa es la ventaja de haber modularizado.

In [47]:
# Comprobación final: lo que se guardó se puede volver a leer sin sorpresas
releido = pd.read_csv(entregables["entrenamiento"])

print("Verificación de ida y vuelta")
print(f"  Antes de guardar : {entrenamiento_final.shape}")
print(f"  Después de leer  : {releido.shape}")
print(f"  Mismas columnas  : {list(releido.columns) == list(entrenamiento_final.columns)}")
print(f"  Sin nulos nuevos : {int(releido.isna().sum().sum()) == int(entrenamiento_final.isna().sum().sum())}")
print("\nEl conjunto está listo para la Fase 4.")

Verificación de ida y vuelta
  Antes de guardar : (11788, 50)
  Después de leer  : (11788, 50)
  Mismas columnas  : True
  Sin nulos nuevos : True

El conjunto está listo para la Fase 4.


---
# 14. Ejercicios: dónde quedó resuelto cada uno

| Ejercicio | Criterio que alimenta | Dónde está resuelto |
|---|---|---|
| 1 · Encapsular un paso del pipeline | Programación orientada a objetos | Sección 5: `ConversorPrecio`, `FiltroFilas`, `SelectorColumnas`, `Acotador`, `MarcadorNulos` y `ExtractorComodidades` |
| 2 · Armar el pipeline completo | Diseño estructurado y preprocesamiento | Sección 6.1: `construir_pipeline()`, los 24 pasos de la Fase 2 |
| 3 · Comparar dos implementaciones | Eficiencia y optimización | Sección 10: escalado con bucle vs vectorizado, con `timeit` y `tracemalloc` |
| 4 · Validar en tres escenarios | Validación técnica | Sección 8: caso normal, casos límite y excepciones |
| 5 · Patrón y equivalencia con la F2 | Documentación de arquitectura y validación técnica | Patrón declarado: sección 11 (Strategy). Equivalencia: sección 8, `assert_frame_equal` contra `airbnb_procesado.csv` |

---
# Cierre

## Lo que se llevan

| Concepto | Dónde se ve en este cuaderno |
|---|---|
| Encapsulamiento | `_parametros`, `_ajustado`, propiedades de solo lectura |
| Herencia | Las clases de pasos (`ImputadorMediana`, `CodificadorNominal`, `EscaladorRobusto`, `FiltroFilas`, etc.) heredan de `Transformador` |
| Polimorfismo | El `Pipeline` recorre los 24 pasos sin preguntar su tipo |
| Recursividad | `aplanar()` y el contraste entre Fibonacci ingenua y memoizada |
| Eficiencia | Escalado con bucle vs vectorizado: `timeit`, `tracemalloc` y la tabla de crecimiento |
| Equivalencia con F2 | `assert_frame_equal` contra `airbnb_procesado.csv` (sección 8) |
| Modularidad | Todo el pipeline se importa desde `src/` (sección 13.3 bis) |
| Patrones | Strategy, Factory, Observer y Singleton |

## Bibliografía (APA 7)

Gamma, E., Helm, R., Johnson, R., & Vlissides, J. (1994). *Design patterns: Elements of reusable object-oriented software*. Addison-Wesley.

McKinney, W. (2022). *Python for data analysis* (3.ª ed.). O'Reilly Media.

Python Software Foundation. (s. f.). *Classes*. https://docs.python.org/3/tutorial/classes.html

Python Software Foundation. (s. f.). *timeit — Measure execution time of small code snippets*. https://docs.python.org/3/library/timeit.html

The pandas development team. (s. f.). *pandas documentation*. https://pandas.pydata.org/docs/